# Spatial / Route Progress: подготовка к CatBoost

Сначала пошагово строим признаки на train, затем повторяем тот же расчёт одной функцией для train и test. Итог — согласованные `X_train`, `y_train`, `X_test`, `y_test` для будущего обучения.

Первые признаки: расстояние до цели, наличие и возраст позиции. Общая база: текущая задержка и оставшееся плановое время. Направления, map matching и ETA отложены до первого сравнения моделей. Здесь CatBoost ещё не обучается.

Офлайн-доступность данных определяем по README: `event_time <= T`. Это не проверка времени фактического получения пакетов в онлайн-системе.

Для воспроизведения выполните **Restart Kernel and Run All Cells**. Исходные CSV не изменяются.

In [1]:
from pathlib import Path


import pandas as pd
from IPython.display import display

DATA_DIR = Path(
    "/Users/vladimir/Desktop/мостранспорт хакаотон/dataset"
)

assert DATA_DIR.exists(), "Папка с датасетом не найдена"

In [2]:
# Задания на прогноз вместе с правильными ответами
labels_train = pd.read_csv(
    DATA_DIR / "labels/labels_train.csv",
    dtype={
        "sample_id": "string",
        "tr_id": "string",
        "target_stop_id": "string",
    },
)

# GPS-телеметрия: пока берём поля для пространственного блока
traffic_train = pd.read_csv(
    DATA_DIR / "train/traffic.csv",
    usecols=[
        "packet_id",
        "tr_id",
        "event_time",
        "location_valid",
        "lon",
        "lat",
        "heading",
    ],
    dtype={
        "packet_id": "string",
        "tr_id": "string",
    },
)

# План остановок: выбираем только нужные плановые поля
plan_train = pd.read_csv(
    DATA_DIR / "train/schedule.csv",
    usecols=[
        "tt_action_item_id",
        "tr_id",
        "time_begin",
        "geom",
        "building_address",
    ],
    dtype={
        "tt_action_item_id": "string",
        "tr_id": "string",
    },
)

превращаем текстовые даты в даты Python

In [3]:
for column in ["T", "target_time_begin"]:
    labels_train[column] = pd.to_datetime(
        labels_train[column],
        format="mixed",
        errors="raise",
    )

traffic_train["event_time"] = pd.to_datetime(
    traffic_train["event_time"],
    format="mixed",
    errors="raise",
)

plan_train["time_begin"] = pd.to_datetime(
    plan_train["time_begin"],
    format="mixed",
    errors="raise",
)

отделяем задания от ответов

In [4]:
# Входная информация о каждом задании на прогноз
samples_train = labels_train[
    [
        "sample_id",
        "tr_id",
        "T",
        "target_stop_id",
        "target_time_begin",
        "cur_dev_s",
    ]
].copy()

# Правильные ответы храним отдельно
targets_train = labels_train[
    [
        "sample_id",
        "target_delay_s",
    ]
].copy()

проверяем загрузку и смотрим данные

In [5]:
assert samples_train["sample_id"].is_unique
assert not samples_train[["T", "target_time_begin"]].isna().any().any()
assert "target_delay_s" not in samples_train.columns
assert "time_fact_begin" not in plan_train.columns

print("Заданий на прогноз:", len(samples_train))
print("Правильных ответов:", len(targets_train))
print("GPS-пакетов:", len(traffic_train))
print("Плановых событий:", len(plan_train))

display(samples_train.head())
display(targets_train.head())
display(traffic_train.head())
display(plan_train.head())

Заданий на прогноз: 4434
Правильных ответов: 4434
GPS-пакетов: 287849
Плановых событий: 16674


,sample_id,tr_id,T,target_stop_id,target_time_begin,cur_dev_s
0,122048_1767665400,122048,2026-01-06 02:10:00,53699018679,2026-01-06 02:22:00,0.0
1,122048_1767665700,122048,2026-01-06 02:15:00,53699018678,2026-01-06 02:26:00,0.0
2,122048_1767666000,122048,2026-01-06 02:20:00,53699018687,2026-01-06 02:32:00,0.0
3,122048_1767666300,122048,2026-01-06 02:25:00,53699018692,2026-01-06 02:37:00,0.0
4,122048_1767666600,122048,2026-01-06 02:30:00,53699018689,2026-01-06 02:41:00,0.0


,sample_id,target_delay_s
0,122048_1767665400,-25.0
1,122048_1767665700,0.0
2,122048_1767666000,0.0
3,122048_1767666300,0.0
4,122048_1767666600,0.0


,packet_id,tr_id,event_time,location_valid,lon,lat,heading
0,-5517330596515664264,115106,2026-01-06 12:30:31.462764,False,NaN,NaN,NaN
1,-5517330596515614428,116057,2026-01-06 12:32:00.058242,False,NaN,NaN,NaN
2,-5517330596515667265,116445,2026-01-06 12:30:26.176861,False,NaN,NaN,NaN
3,-5517330596515596171,118726,2026-01-06 12:32:31.682087,False,NaN,NaN,NaN
4,-5517330596515637929,119145,2026-01-06 12:31:18.279263,False,NaN,NaN,NaN


,tt_action_item_id,time_begin,tr_id,geom,building_address
0,53699433970,2026-01-06 06:36:00,122658,POINT (37.43070705 55.8040083),"Строгинское ш., д.1"
1,53699433974,2026-01-06 06:41:00,122658,POINT (37.46330869 55.80892705),"ул. Маршала Василевского, д.17"
2,53699433972,2026-01-06 06:38:00,122658,POINT (37.45439344 55.80550491),"Новощукинская ул., д.7"
3,53699456537,2026-01-06 13:05:00,133957,POINT (37.53020277 55.64161975),"ул. Миклухо-Маклая, д.53, к.1"
4,53700172912,2026-01-06 07:47:00,131672,POINT (37.85747284 55.73916511),"Новокосинская ул., д.17, к.3"


In [6]:
# Извлекаем два числа из строки POINT (...)
coordinates = plan_train["geom"].str.extract(
    r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)"
)

# Добавляем их в расписание как числовые колонки
plan_train["stop_lon"] = coordinates[0].astype(float)
plan_train["stop_lat"] = coordinates[1].astype(float)

# Проверяем, что координаты извлечены и находятся в допустимых пределах
assert plan_train["stop_lon"].between(-180, 180).all()
assert plan_train["stop_lat"].between(-90, 90).all()

display(
    plan_train[
        ["tt_action_item_id", "geom", "stop_lon", "stop_lat"]
    ].head()
)

,tt_action_item_id,geom,stop_lon,stop_lat
0,53699433970,POINT (37.43070705 55.8040083),37.430707,55.804008
1,53699433974,POINT (37.46330869 55.80892705),37.463309,55.808927
2,53699433972,POINT (37.45439344 55.80550491),37.454393,55.805505
3,53699456537,POINT (37.53020277 55.64161975),37.530203,55.641620
4,53700172912,POINT (37.85747284 55.73916511),37.857473,55.739165


Подготавливаем таблицу для присоединения

In [7]:
target_stops = plan_train[
    [
        "tr_id",
        "tt_action_item_id",
        "time_begin",
        "stop_lon",
        "stop_lat",
    ]
].rename(
    columns={
        "tt_action_item_id": "target_stop_id",
        "time_begin": "target_plan_time",
        "stop_lon": "target_lon",
        "stop_lat": "target_lat",
    }
)

display(target_stops.head())

,tr_id,target_stop_id,target_plan_time,target_lon,target_lat
0,122658,53699433970,2026-01-06 06:36:00,37.430707,55.804008
1,122658,53699433974,2026-01-06 06:41:00,37.463309,55.808927
2,122658,53699433972,2026-01-06 06:38:00,37.454393,55.805505
3,133957,53699456537,2026-01-06 13:05:00,37.530203,55.641620
4,131672,53700172912,2026-01-06 07:47:00,37.857473,55.739165


In [8]:
samples_geo = samples_train.merge(
    target_stops,
    on=["tr_id", "target_stop_id"],
    how="left",
    validate="many_to_one",
    indicator="_target_join",
)

In [9]:
# После соединения количество заданий не изменилось
assert len(samples_geo) == len(samples_train)
assert samples_geo["sample_id"].is_unique

# Для каждого задания нашлась цель
assert samples_geo["_target_join"].eq("both").all()

# Координаты присутствуют
assert samples_geo[["target_lon", "target_lat"]].notna().all().all()

# Время из задания совпадает со временем из расписания
assert samples_geo["target_time_begin"].eq(
    samples_geo["target_plan_time"]
).all()

# Проверки завершены — убираем служебные колонки
samples_geo = samples_geo.drop(
    columns=["_target_join", "target_plan_time"]
)

print("Заданий с координатами цели:", len(samples_geo))

display(
    samples_geo[
        [
            "sample_id",
            "tr_id",
            "T",
            "target_stop_id",
            "target_time_begin",
            "target_lon",
            "target_lat",
        ]
    ].head()
)

Заданий с координатами цели: 4434


,sample_id,tr_id,T,target_stop_id,target_time_begin,target_lon,target_lat
0,122048_1767665400,122048,2026-01-06 02:10:00,53699018679,2026-01-06 02:22:00,37.748513,55.613893
1,122048_1767665700,122048,2026-01-06 02:15:00,53699018678,2026-01-06 02:26:00,37.743482,55.620128
2,122048_1767666000,122048,2026-01-06 02:20:00,53699018687,2026-01-06 02:32:00,37.727965,55.621924
3,122048_1767666300,122048,2026-01-06 02:25:00,53699018692,2026-01-06 02:37:00,37.712607,55.624677
4,122048_1767666600,122048,2026-01-06 02:30:00,53699018689,2026-01-06 02:41:00,37.694534,55.636919


Теперь добавим к каждому заданию последнее корректное положение машины на момент T.

Оставляем пакеты с корректными координатами

In [10]:
valid_position = (
    traffic_train["location_valid"].eq(True)
    & traffic_train["lon"].between(-180, 180)
    & traffic_train["lat"].between(-90, 90)
    & ~(
        traffic_train["lon"].eq(0)
        & traffic_train["lat"].eq(0)
    )
)

traffic_valid = traffic_train.loc[valid_position].copy()

print("Всего пакетов:", len(traffic_train))
print("С корректными координатами:", len(traffic_valid))

Всего пакетов: 287849
С корректными координатами: 243974



Подготавливаем таблицы к поиску по времени

In [11]:
positions = traffic_valid[
    [
        "packet_id",
        "tr_id",
        "event_time",
        "lon",
        "lat",
        "heading",
    ]
].rename(
    columns={
        "event_time": "position_time",
        "lon": "current_lon",
        "lat": "current_lat",
        "heading": "current_heading",
    }
)

# В первую очередь сортируем по времени
positions = positions.sort_values(
    ["position_time", "tr_id", "packet_id"]
)

# Если у машины несколько пакетов с одинаковым временем,
# выбираем один по фиксированному правилу
positions = positions.drop_duplicates(
    subset=["tr_id", "position_time"],
    keep="last",
)

samples_sorted = samples_geo.sort_values(
    ["T", "sample_id"]
)


Для каждого задания находим последнюю прошлую позицию

In [12]:
samples_position = pd.merge_asof(
    samples_sorted,
    positions,
    left_on="T",
    right_on="position_time",
    by="tr_id",
    direction="backward",
    allow_exact_matches=True,
)

Добавляем первые признаки: наличие и возраст позиции

In [13]:
samples_position["sp_has_position"] = (
    samples_position["position_time"].notna().astype(int)
)

samples_position["sp_position_age_s"] = (
    samples_position["T"] - samples_position["position_time"]
).dt.total_seconds()

проверяем результат

In [14]:
# Все задания сохранились
assert len(samples_position) == len(samples_geo)
assert samples_position["sample_id"].is_unique

# Ни одна найденная позиция не относится к будущему
has_position = samples_position["position_time"].notna()

assert (
    samples_position.loc[has_position, "position_time"]
    <= samples_position.loc[has_position, "T"]
).all()

print("Всего заданий:", len(samples_position))
print("Без прошлой корректной позиции:", (~has_position).sum())

display(
    samples_position[
        [
            "sample_id",
            "tr_id",
            "T",
            "position_time",
            "current_lon",
            "current_lat",
            "target_lon",
            "target_lat",
            "sp_position_age_s",
        ]
    ].head()
)

Всего заданий: 4434
Без прошлой корректной позиции: 1


,sample_id,tr_id,T,position_time,current_lon,current_lat,target_lon,target_lat,sp_position_age_s
0,122048_1767665400,122048,2026-01-06 02:10:00,2026-01-06 02:09:52.000000000,37.747913,55.613964,37.748513,55.613893,8.000000
1,9000001_1767665400,9000001,2026-01-06 02:10:00,2026-01-06 02:09:58.862763477,37.748039,55.613765,37.748513,55.613893,1.137237
2,9000014_1767665400,9000014,2026-01-06 02:10:00,NaT,NaN,NaN,37.480842,55.598864,NaN
3,122048_1767665700,122048,2026-01-06 02:15:00,2026-01-06 02:14:54.000000000,37.747887,55.613792,37.743482,55.620128,6.000000
4,9000001_1767665700,9000001,2026-01-06 02:15:00,2026-01-06 02:14:46.862763477,37.748007,55.613866,37.744328,55.614880,13.137237


**Теперь считаем расстояние от последней известной позиции машины до целевой остановки**

функция расчета расстояния

In [15]:
import numpy as np


def haversine_m(lon1, lat1, lon2, lat2):
    """Расстояние между координатами в метрах по сфере."""
    earth_radius_m = 6_371_008.8

    # Переводим градусы в радианы
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    delta_lon = lon2 - lon1
    delta_lat = lat2 - lat1

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(delta_lon / 2) ** 2
    )

    # Защита от небольших погрешностей вычислений
    a = np.clip(a, 0, 1)

    return 2 * earth_radius_m * np.arcsin(np.sqrt(a))

Считаем расстояние для всех заданий

In [16]:
samples_position["sp_distance_to_target_m"] = haversine_m(
    samples_position["current_lon"],
    samples_position["current_lat"],
    samples_position["target_lon"],
    samples_position["target_lat"],
)

display(
    samples_position[
        [
            "sample_id",
            "sp_position_age_s",
            "sp_distance_to_target_m",
        ]
    ].head(10)
)

,sample_id,sp_position_age_s,sp_distance_to_target_m
0,122048_1767665400,8.000000,38.518950
1,9000001_1767665400,1.137237,32.948434
2,9000014_1767665400,NaN,NaN
3,122048_1767665700,6.000000,756.878020
4,9000001_1767665700,13.137237,257.054020
5,9000014_1767665700,0.123413,5785.359469
6,9000021_1767665700,11.833071,26.475305
7,122048_1767666000,2.000000,1536.898448
8,9000001_1767666000,11.137237,1146.791406
9,9000007_1767666000,3.094006,87.739285


Проверяем полученную колонку

In [17]:
distance = samples_position["sp_distance_to_target_m"]

# Все рассчитанные расстояния неотрицательны и конечны
assert distance.dropna().ge(0).all()
assert np.isfinite(distance.dropna()).all()

# Если позиции нет, расстояние должно остаться неизвестным
no_position = samples_position["sp_has_position"].eq(0)
assert distance.loc[no_position].isna().all()

# Если позиция есть, при найденных координатах цели расстояние рассчитано
assert distance.loc[~no_position].notna().all()

print("Заданий:", len(samples_position))
print("Расстояние рассчитано:", distance.notna().sum())
print("Расстояние неизвестно:", distance.isna().sum())

display(
    distance.describe(percentiles=[0.5, 0.9, 0.95])
)

Заданий: 4434
Расстояние рассчитано: 4433
Расстояние неизвестно: 1


count     4433.000000
mean      2988.768878
std       3910.947315
min          3.355516
50%       1975.022832
90%       5272.944719
95%       8356.312468
max      33147.328189
Name: sp_distance_to_target_m, dtype: float64

## 1. Завершаем текущую таблицу: оставшееся плановое время

`prediction_horizon_s` — через сколько секунд по плану машина должна прибыть на цель. Это известная входная информация, не фактическое будущее прибытие. По условию горизонт должен быть больше 600 и не больше 900 секунд.

In [18]:
samples_position["prediction_horizon_s"] = (
    samples_position["target_time_begin"] - samples_position["T"]
).dt.total_seconds()

assert samples_position["prediction_horizon_s"].gt(600).all()
assert samples_position["prediction_horizon_s"].le(900).all()

display(samples_position[
    ["sample_id", "cur_dev_s", "prediction_horizon_s",
     "sp_has_position", "sp_position_age_s", "sp_distance_to_target_m"]
].head())

,sample_id,cur_dev_s,prediction_horizon_s,sp_has_position,sp_position_age_s,sp_distance_to_target_m
0,122048_1767665400,0.0,720.000000,1,8.000000,38.518950
1,9000001_1767665400,0.0,786.862763,1,1.137237,32.948434
2,9000014_1767665400,0.0,656.876587,0,NaN,NaN
3,122048_1767665700,0.0,660.000000,1,6.000000,756.878020
4,9000001_1767665700,0.0,606.862763,1,13.137237,257.054020


## 2. Тот же расчёт в одной функции

Учебные ячейки выше оставлены. Функция ниже повторяет их логику, чтобы train и test обрабатывались одинаково. На вход ей передаются **задания без ответов**, GPS и план без фактического времени прибытия.

Функция возвращает строки в исходном порядке заданий. Тем не менее ответы далее присоединяем явно по `sample_id`, чтобы сортировка никогда не перепутала признаки и ответы.

In [19]:
SAMPLE_COLUMNS = [
    "sample_id", "tr_id", "T", "target_stop_id",
    "target_time_begin", "cur_dev_s",
]
TRAFFIC_COLUMNS = [
    "packet_id", "tr_id", "event_time", "location_valid",
    "lon", "lat", "heading",
]
PLAN_COLUMNS = [
    "tt_action_item_id", "tr_id", "time_begin", "geom", "building_address",
]
BASE_FEATURE_COLUMNS = ["cur_dev_s", "prediction_horizon_s"]
SPATIAL_FEATURE_COLUMNS = [
    "sp_has_position", "sp_position_age_s", "sp_distance_to_target_m",
]

APPROACH_FEATURE_COLUMNS = [
    "sp_approach_1m_m",
    "sp_approach_speed_1m_mps",
    "sp_approach_3m_m",
    "sp_approach_speed_3m_mps",
]

FEATURE_COLUMNS = (
    BASE_FEATURE_COLUMNS
    + SPATIAL_FEATURE_COLUMNS
    + APPROACH_FEATURE_COLUMNS
)


def build_spatial_features(samples, traffic, plan):
    """Повторяет расчёт выше; использует только пакеты event_time <= T."""
    samples = samples[SAMPLE_COLUMNS].copy()
    traffic = traffic[TRAFFIC_COLUMNS].copy()
    plan = plan[PLAN_COLUMNS].copy()

    assert samples["sample_id"].is_unique
    assert samples[["sample_id", "tr_id", "target_stop_id"]].notna().all().all()

    for column in ["T", "target_time_begin"]:
        samples[column] = pd.to_datetime(samples[column], format="mixed", errors="raise")
    traffic["event_time"] = pd.to_datetime(traffic["event_time"], format="mixed", errors="raise")
    plan["time_begin"] = pd.to_datetime(plan["time_begin"], format="mixed", errors="raise")
    assert samples[["T", "target_time_begin"]].notna().all().all()
    assert traffic["event_time"].notna().all()

    coordinates = plan["geom"].str.extract(
        r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)"
    ).astype(float)
    plan["target_lon"] = coordinates[0]
    plan["target_lat"] = coordinates[1]
    assert plan["target_lon"].between(-180, 180).all()
    assert plan["target_lat"].between(-90, 90).all()

    stops = plan[
        ["tr_id", "tt_action_item_id", "time_begin", "target_lon", "target_lat"]
    ].rename(columns={
        "tt_action_item_id": "target_stop_id",
        "time_begin": "target_plan_time",
    })
    result = samples.merge(
        stops, on=["tr_id", "target_stop_id"], how="left",
        validate="many_to_one", indicator="_target_join",
    )
    assert result["_target_join"].eq("both").all(), "Не найдена геометрия цели"
    assert result["target_time_begin"].eq(result["target_plan_time"]).all()
    result = result.drop(columns=["_target_join", "target_plan_time"])

    valid = (
        traffic["location_valid"].eq(True)
        & traffic["lon"].between(-180, 180)
        & traffic["lat"].between(-90, 90)
        & ~(traffic["lon"].eq(0) & traffic["lat"].eq(0))
    )
    positions = traffic.loc[valid, [
        "packet_id", "tr_id", "event_time", "lon", "lat", "heading"
    ]].rename(columns={
        "event_time": "position_time", "lon": "current_lon",
        "lat": "current_lat", "heading": "current_heading",
    })
    positions = positions.sort_values(["position_time", "tr_id", "packet_id"])
    positions = positions.drop_duplicates(["tr_id", "position_time"], keep="last")
    result = pd.merge_asof(
        result.sort_values(["T", "sample_id"]), positions,
        left_on="T", right_on="position_time", by="tr_id",
        direction="backward", allow_exact_matches=True,
    )

    result["sp_has_position"] = result["position_time"].notna().astype(int)
    result["sp_position_age_s"] = (result["T"] - result["position_time"]).dt.total_seconds()
    result["sp_distance_to_target_m"] = haversine_m(
        result["current_lon"], result["current_lat"],
        result["target_lon"], result["target_lat"],
    )
    result["prediction_horizon_s"] = (
        result["target_time_begin"] - result["T"]
    ).dt.total_seconds()

        # Приближение к цели за 1 и 3 минуты.
    for minutes in [1, 3]:
        query_time = f"time_{minutes}m_ago"
        past_time = f"position_time_{minutes}m_ago"
        past_lon = f"lon_{minutes}m_ago"
        past_lat = f"lat_{minutes}m_ago"

        result[query_time] = (
            result["T"] - pd.Timedelta(minutes=minutes)
        )

        past_positions = positions[
            ["tr_id", "position_time", "current_lon", "current_lat"]
        ].rename(columns={
            "position_time": past_time,
            "current_lon": past_lon,
            "current_lat": past_lat,
        })

        result = pd.merge_asof(
            result.sort_values(query_time),
            past_positions.sort_values(past_time),
            left_on=query_time,
            right_on=past_time,
            by="tr_id",
            direction="backward",
            tolerance=pd.Timedelta(seconds=60),
        )

        found = result[past_time].notna()
        assert (
            result.loc[found, past_time]
            <= result.loc[found, query_time]
        ).all()

        past_distance = haversine_m(
            result[past_lon],
            result[past_lat],
            result["target_lon"],
            result["target_lat"],
        )

        elapsed_s = (
            result["position_time"] - result[past_time]
        ).dt.total_seconds()

        usable = (
            result["sp_position_age_s"].between(0, 60)
            & elapsed_s.gt(0)
            & past_distance.notna()
            & result["sp_distance_to_target_m"].notna()
        )

        distance_column = f"sp_approach_{minutes}m_m"
        speed_column = f"sp_approach_speed_{minutes}m_mps"

        result[distance_column] = (
            past_distance - result["sp_distance_to_target_m"]
        ).where(usable)

        result[speed_column] = (
            result[distance_column] / elapsed_s.where(usable)
        )
        
    assert len(result) == len(samples) and result["sample_id"].is_unique
    assert result["sp_position_age_s"].dropna().ge(0).all()
    assert result["sp_distance_to_target_m"].dropna().ge(0).all()
    assert result["prediction_horizon_s"].gt(600).all()
    assert result["prediction_horizon_s"].le(900).all()
    assert not np.isinf(result[FEATURE_COLUMNS].to_numpy(dtype=float)).any()
    has_position = result["sp_has_position"].eq(1)
    position_features = ["sp_position_age_s", "sp_distance_to_target_m"]
    assert result.loc[has_position, position_features].notna().all().all()
    assert result.loc[~has_position, position_features].isna().all().all()

    return result.set_index("sample_id").loc[samples["sample_id"]].reset_index()

In [20]:
# Для каждого задания задаём моменты поиска прошлых позиций.
history_queries = samples_train[
    ["sample_id", "tr_id", "T"]
].copy()

history_queries["T"] = pd.to_datetime(
    history_queries["T"],
    format="mixed",
    errors="raise",
)

history_queries["time_1m_ago"] = (
    history_queries["T"] - pd.Timedelta(minutes=1)
)

history_queries["time_3m_ago"] = (
    history_queries["T"] - pd.Timedelta(minutes=3)
)

display(history_queries.head())

,sample_id,tr_id,T,time_1m_ago,time_3m_ago
0,122048_1767665400,122048,2026-01-06 02:10:00,2026-01-06 02:09:00,2026-01-06 02:07:00
1,122048_1767665700,122048,2026-01-06 02:15:00,2026-01-06 02:14:00,2026-01-06 02:12:00
2,122048_1767666000,122048,2026-01-06 02:20:00,2026-01-06 02:19:00,2026-01-06 02:17:00
3,122048_1767666300,122048,2026-01-06 02:25:00,2026-01-06 02:24:00,2026-01-06 02:22:00
4,122048_1767666600,122048,2026-01-06 02:30:00,2026-01-06 02:29:00,2026-01-06 02:27:00


In [21]:
history_positions = history_queries.copy()

for minutes in [1, 3]:
    query_time = f"time_{minutes}m_ago"
    past_time = f"position_time_{minutes}m_ago"

    past_positions = positions[
        ["tr_id", "position_time", "current_lon", "current_lat"]
    ].rename(columns={
        "position_time": past_time,
        "current_lon": f"lon_{minutes}m_ago",
        "current_lat": f"lat_{minutes}m_ago",
    })

    history_positions = pd.merge_asof(
        history_positions.sort_values(query_time),
        past_positions.sort_values(past_time),
        left_on=query_time,
        right_on=past_time,
        by="tr_id",
        direction="backward",
        tolerance=pd.Timedelta(seconds=60),
    )

    found = history_positions[past_time].notna()

    assert (
        history_positions.loc[found, past_time]
        <= history_positions.loc[found, query_time]
    ).all()

    print(
        f"Позиция {minutes} мин назад найдена:",
        f"{found.sum()} из {len(history_positions)}",
    )

assert len(history_positions) == len(history_queries)
assert history_positions["sample_id"].is_unique

display(
    history_positions[
        [
            "sample_id", "T",
            "position_time_1m_ago", "lon_1m_ago", "lat_1m_ago",
            "position_time_3m_ago", "lon_3m_ago", "lat_3m_ago",
        ]
    ].head()
)

Позиция 1 мин назад найдена: 4037 из 4434
Позиция 3 мин назад найдена: 4022 из 4434


,sample_id,T,position_time_1m_ago,lon_1m_ago,lat_1m_ago,position_time_3m_ago,lon_3m_ago,lat_3m_ago
0,122048_1767665400,2026-01-06 02:10:00,2026-01-06 02:08:52.000000000,37.747917,55.613820,2026-01-06 02:06:56.000000000,37.747986,55.613850
1,9000014_1767665400,2026-01-06 02:10:00,NaT,NaN,NaN,NaT,NaN,NaN
2,9000001_1767665400,2026-01-06 02:10:00,2026-01-06 02:08:48.862763477,37.747857,55.613844,2026-01-06 02:06:46.862763477,37.747952,55.613889
3,122048_1767665700,2026-01-06 02:15:00,2026-01-06 02:13:54.000000000,37.747944,55.613870,2026-01-06 02:11:46.000000000,37.747990,55.613840
4,9000014_1767665700,2026-01-06 02:15:00,2026-01-06 02:13:53.876587200,37.504447,55.619000,NaT,NaN,NaN


In [22]:
# Соединяем по sample_id: порядок строк после сортировок мог измениться.
history_columns = [
    "sample_id",
    "position_time_1m_ago", "lon_1m_ago", "lat_1m_ago",
    "position_time_3m_ago", "lon_3m_ago", "lat_3m_ago",
]

approach_train = samples_position.merge(
    history_positions[history_columns],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

APPROACH_FEATURE_COLUMNS = []

for minutes in [1, 3]:
    # Расстояние от прошлой позиции до той же цели текущего задания.
    past_distance = haversine_m(
        approach_train[f"lon_{minutes}m_ago"],
        approach_train[f"lat_{minutes}m_ago"],
        approach_train["target_lon"],
        approach_train["target_lat"],
    )

    # Реальное время между текущей и прошлой GPS-позициями.
    elapsed_s = (
        approach_train["position_time"]
        - approach_train[f"position_time_{minutes}m_ago"]
    ).dt.total_seconds()

    # Текущая позиция должна быть свежей и новее прошлой.
    usable = (
        approach_train["sp_position_age_s"].between(0, 60)
        & elapsed_s.gt(0)
        & past_distance.notna()
        & approach_train["sp_distance_to_target_m"].notna()
    )

    distance_column = f"sp_approach_{minutes}m_m"
    speed_column = f"sp_approach_speed_{minutes}m_mps"

    approach_train[distance_column] = (
        past_distance - approach_train["sp_distance_to_target_m"]
    ).where(usable)

    approach_train[speed_column] = (
        approach_train[distance_column] / elapsed_s.where(usable)
    )

    APPROACH_FEATURE_COLUMNS.extend([
        distance_column,
        speed_column,
    ])

assert len(approach_train) == len(samples_position)
assert not np.isinf(
    approach_train[APPROACH_FEATURE_COLUMNS].to_numpy(dtype=float)
).any()

display(
    approach_train[
        ["sample_id"] + APPROACH_FEATURE_COLUMNS
    ].head(10)
)

print("Количество рассчитанных значений:")
display(approach_train[APPROACH_FEATURE_COLUMNS].count())

,sample_id,sp_approach_1m_m,sp_approach_speed_1m_mps,sp_approach_3m_m,sp_approach_speed_3m_mps
0,122048_1767665400,-0.221043,-0.003684,-5.076956,-0.028846
1,9000001_1767665400,8.613605,0.123052,2.301896,0.011989
2,9000014_1767665400,NaN,NaN,NaN,NaN
3,122048_1767665700,-6.737204,-0.112287,-2.562494,-0.013630
4,9000001_1767665700,-3.529525,-0.058825,-2.068099,-0.011886
5,9000014_1767665700,155.641207,2.358200,NaN,NaN
6,9000021_1767665700,13.357094,0.222618,NaN,NaN
7,122048_1767666000,2.338342,0.037715,5.196934,0.028244
8,9000001_1767666000,-23.544139,-0.379744,-18.017656,-0.098998
9,9000007_1767666000,-1.224682,-0.020411,16.960075,0.094223


Количество рассчитанных значений:


sp_approach_1m_m            4006
sp_approach_speed_1m_mps    4006
sp_approach_3m_m            3958
sp_approach_speed_3m_mps    3958
dtype: int64

In [23]:
# Проверяем, что единая функция воспроизводит уже выполненные учебные ячейки.
features_train = build_spatial_features(samples_train, traffic_train, plan_train)

# В samples_position находятся только первоначальные признаки.
original_columns = BASE_FEATURE_COLUMNS + SPATIAL_FEATURE_COLUMNS

pd.testing.assert_frame_equal(
    features_train.set_index("sample_id")[original_columns].sort_index(),
    samples_position.set_index("sample_id")[original_columns].sort_index(),
)

print("Первоначальные признаки совпали с пошаговым расчётом.")

Первоначальные признаки совпали с пошаговым расчётом.


In [24]:
pd.testing.assert_frame_equal(
    features_train.set_index("sample_id")[
        APPROACH_FEATURE_COLUMNS
    ].sort_index(),
    approach_train.set_index("sample_id")[
        APPROACH_FEATURE_COLUMNS
    ].sort_index(),
)

print("Все четыре признака совпали с пошаговым расчётом.")

Все четыре признака совпали с пошаговым расчётом.


## 3. Подготавливаем test тем же способом

На train будем обучать CatBoost, на test — оценивать. Сейчас только рассчитываем входные признаки. `target_delay_s` и `target_class` не передаются в функцию, а `time_fact_begin` вообще не загружается в план.

In [25]:
labels_test = pd.read_csv(
    DATA_DIR / "labels/labels_test.csv",
    dtype={"sample_id": "string", "tr_id": "string", "target_stop_id": "string"},
)
traffic_test = pd.read_csv(
    DATA_DIR / "test/traffic.csv", usecols=TRAFFIC_COLUMNS,
    dtype={"packet_id": "string", "tr_id": "string"},
)
plan_test = pd.read_csv(
    DATA_DIR / "test/schedule.csv", usecols=PLAN_COLUMNS,
    dtype={"tt_action_item_id": "string", "tr_id": "string"},
)

samples_test = labels_test[SAMPLE_COLUMNS].copy()
targets_test = labels_test[["sample_id", "target_delay_s"]].copy()
features_test = build_spatial_features(samples_test, traffic_test, plan_test)

print("Train:", len(features_train), "заданий")
print("Test:", len(features_test), "заданий")
display(features_test[["sample_id"] + FEATURE_COLUMNS].head())

Train: 4434 заданий
Test: 353 заданий


,sample_id,cur_dev_s,prediction_horizon_s,sp_has_position,sp_position_age_s,sp_distance_to_target_m,sp_approach_1m_m,sp_approach_speed_1m_mps,sp_approach_3m_m,sp_approach_speed_3m_mps
0,129964_1767665100,0.0,780.0,1,11.0,92.742876,0.000000,0.000000,0.000000,0.000000
1,129964_1767665400,0.0,660.0,1,11.0,794.012531,-1.511521,-0.025192,-1.511521,-0.008397
2,129964_1767665700,0.0,720.0,1,11.0,1124.881976,0.000000,0.000000,0.000000,0.000000
3,129964_1767666000,-58.0,660.0,1,11.0,1030.675066,-208.990025,-3.483167,-137.489408,-0.763830
4,129964_1767666300,-49.0,660.0,1,11.0,1482.328696,-34.402976,-0.573383,85.763120,0.476462


## 4. Привязываем правильные ответы и готовим X / y

`X` — пять выбранных входных признаков, `y` — задержка в секундах. Соединение идёт по `sample_id`, а не по порядку строк. Сам идентификатор остаётся индексом и не передаётся модели как признак.

Числовые пропуски GPS сохраняем: неизвестное расстояние не равно нулю. Проверка и настройка модели будут отдельным следующим этапом.

In [26]:
def attach_targets(features, targets):
    """Однозначно сопоставляет каждое задание с его правильным ответом."""
    assert features["sample_id"].is_unique and targets["sample_id"].is_unique
    assert set(features["sample_id"]) == set(targets["sample_id"])
    table = features.merge(
        targets[["sample_id", "target_delay_s"]], on="sample_id",
        how="left", validate="one_to_one", indicator="_label_join",
    )
    assert table["_label_join"].eq("both").all()
    assert table["target_delay_s"].notna().all()
    assert np.isfinite(table["target_delay_s"].to_numpy(dtype=float)).all()
    return table.drop(columns="_label_join").set_index("sample_id")


train_table = attach_targets(features_train, targets_train)
test_table = attach_targets(features_test, targets_test)

X_train = train_table[FEATURE_COLUMNS].astype(float).copy()
y_train = train_table["target_delay_s"].astype(float).copy()
X_test = test_table[FEATURE_COLUMNS].astype(float).copy()
y_test = test_table["target_delay_s"].astype(float).copy()

# Общая база для сравнения с моделью, использующей пространственные фичи.
X_train_base = X_train[BASE_FEATURE_COLUMNS].copy()
X_test_base = X_test[BASE_FEATURE_COLUMNS].copy()

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)
assert set(FEATURE_COLUMNS).isdisjoint({
    "sample_id", "tr_id", "packet_id", "target_stop_id",
    "target_delay_s", "target_class", "time_fact_begin",
})

display(pd.DataFrame({
    "выборка": ["train", "test"],
    "заданий": [len(X_train), len(X_test)],
    "входных признаков": [X_train.shape[1], X_test.shape[1]],
    "без GPS": [int(X_train["sp_has_position"].eq(0).sum()),
                int(X_test["sp_has_position"].eq(0).sum())],
}))
display(X_train.head())
print("Подготовка завершена. X_train, y_train, X_test, y_test готовы.")
print("CatBoost ещё не обучался; метрики модели пока нет.")

,выборка,заданий,входных признаков,без GPS
0,train,4434,9,1
1,test,353,9,0


,cur_dev_s,prediction_horizon_s,sp_has_position,sp_position_age_s,sp_distance_to_target_m,sp_approach_1m_m,sp_approach_speed_1m_mps,sp_approach_3m_m,sp_approach_speed_3m_mps
sample_id,,,,,,,,,
122048_1767665400,0.0,720.0,1.0,8.0,38.518950,-0.221043,-0.003684,-5.076956,-0.028846
122048_1767665700,0.0,660.0,1.0,6.0,756.878020,-6.737204,-0.112287,-2.562494,-0.013630
122048_1767666000,0.0,720.0,1.0,2.0,1536.898448,2.338342,0.037715,5.196934,0.028244
122048_1767666300,0.0,720.0,1.0,6.0,2165.018524,93.365196,1.556087,323.237327,1.795763
122048_1767666600,0.0,660.0,1.0,6.0,2887.085352,165.052604,2.750877,668.768604,3.715381


Подготовка завершена. X_train, y_train, X_test, y_test готовы.
CatBoost ещё не обучался; метрики модели пока нет.


## Следующий этап: CatBoost

Готовы две конфигурации: общая база (`X_train_base`) и база с пространственными признаками (`X_train`). Следом обучим `CatBoostRegressor` на train и сравним MAE на test. Число итераций для первого опыта заранее фиксируем либо выбираем внутри train; test не используем для скрытой настройки.

Дополнительную геометрию, направления и ETA будем проверять после первого результата. `validate` пока не обрабатывали, прогнозов для платформы нет.

In [27]:
%pip install catboost scikit-learn

Note: you may need to restart the kernel to use updated packages.


Импортируем инструменты и проверяем таблицы

In [28]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Признаки:", X_train.columns.tolist())

Train: (4434, 9)
Test: (353, 9)
Признаки: ['cur_dev_s', 'prediction_horizon_s', 'sp_has_position', 'sp_position_age_s', 'sp_distance_to_target_m', 'sp_approach_1m_m', 'sp_approach_speed_1m_mps', 'sp_approach_3m_m', 'sp_approach_speed_3m_mps']


Задаём настройки первого эксперимента

In [29]:
model_params = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.05,
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "random_seed": 42,
    "thread_count": 4,
    "allow_writing_files": False,
    "verbose": 100,
}

4. Обучаем базовый CatBoost

In [30]:
model_base = CatBoostRegressor(**model_params)

model_base.fit(
    X_train_base,
    y_train,
)

0:	learn: 93.2276490	total: 59.5ms	remaining: 29.7s
100:	learn: 78.5847980	total: 90.5ms	remaining: 358ms
200:	learn: 77.1966987	total: 121ms	remaining: 180ms
300:	learn: 76.4385359	total: 150ms	remaining: 99.4ms
400:	learn: 75.9007140	total: 180ms	remaining: 44.5ms
499:	learn: 74.8435651	total: 210ms	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=5, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='MAE', random_seed=42, thread_count=4, verbose=100)

Обучаем CatBoost с твоими признаками

In [31]:
model_spatial = CatBoostRegressor(**model_params)

model_spatial.fit(
    X_train,
    y_train,
)

0:	learn: 93.3573400	total: 499us	remaining: 249ms
100:	learn: 73.7769783	total: 39.8ms	remaining: 157ms
200:	learn: 70.4433959	total: 79.3ms	remaining: 118ms
300:	learn: 67.6118152	total: 119ms	remaining: 78.4ms
400:	learn: 65.6641312	total: 158ms	remaining: 39ms
499:	learn: 64.1762024	total: 197ms	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=5, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='MAE', random_seed=42, thread_count=4, verbose=100)

Получаем прогнозы на test и сравниваем MAE

In [32]:
# Прогнозы двух обученных моделей
prediction_base = model_base.predict(X_test_base)
prediction_spatial = model_spatial.predict(X_test)

# Простой ориентир: будущая задержка равна текущей
prediction_current = X_test["cur_dev_s"].to_numpy()

mae_current = mean_absolute_error(y_test, prediction_current)
mae_base = mean_absolute_error(y_test, prediction_base)
mae_spatial = mean_absolute_error(y_test, prediction_spatial)

results = pd.DataFrame({
    "Вариант": [
        "Ориентир: текущая задержка",
        "CatBoost: общая база",
        "CatBoost: база + spatial",
    ],
    "MAE, секунды": [mae_current, mae_base, mae_spatial],
})

display(results.round(2))
print(
    "Улучшение от spatial относительно базового CatBoost:",
    round(mae_base - mae_spatial, 2),
    "секунды",
)

,Вариант,"MAE, секунды"
0,Ориентир: текущая задержка,93.36
1,CatBoost: общая база,84.21
2,CatBoost: база + spatial,75.36


Улучшение от spatial относительно базового CatBoost: 8.85 секунды


Посмотрим несколько прогнозов

In [33]:
test_predictions = pd.DataFrame({
    "actual_delay_s": y_test,
    "prediction_base_s": prediction_base,
    "prediction_spatial_s": prediction_spatial,
})

test_predictions["absolute_error_spatial_s"] = (
    test_predictions["actual_delay_s"]
    - test_predictions["prediction_spatial_s"]
).abs()

display(test_predictions.head(10).round(1))

,actual_delay_s,prediction_base_s,prediction_spatial_s,absolute_error_spatial_s
sample_id,,,,
129964_1767665100,-45.0,6.1,36.1,81.1
129964_1767665400,-37.0,0.0,45.1,82.1
129964_1767665700,-63.0,0.0,39.4,102.4
129964_1767666000,-113.0,-26.0,-1.5,111.5
129964_1767666300,-3.0,-17.0,-18.0,15.0
131672_1767666900,79.0,5.9,-8.0,87.0
131672_1767667200,79.0,0.0,28.4,50.6
131672_1767667500,212.0,0.0,43.9,168.1
131672_1767667800,312.0,71.1,85.4,226.6
